In [1]:
%reset -f
import math
import os
import warnings
from collections.abc import Callable
from math import pow, sqrt

import forallpeople as units
import handcalcs
import handcalcs.render
import numpy as np

# import polars as pl
import pandas as pd
import sympy as sp
from IPython.display import Latex, display
from pandas import IndexSlice as idx

from structural_tools.notebook import (
    check_value,
    display_table,
    display_text,
    set_params_columns,
    sig_figs,
)
from structural_tools.seismic.asce.parameters import (
    RiskCategory,
    SeismicParameters,
    SiteClass,
)

units.environment(env_name="structural", top_level=True)
handcalcs.set_option("custom_symbols", {"star": "^*", "__": ","})
handcalcs.set_option("greek_exclusions", ["psi"])
handcalcs.set_option("display_precision", 4)
handcalcs.set_option("param_columns", 1)
handcalcs.set_option("latex_block_start", "\\begin{equation*}")
handcalcs.set_option("latex_block_end", "\\end{equation*}")

# Effective Seismic Weight Calculation

We calculate the effective seismic weight with unit weights and square footages.

## Unit Weights

Weights will be assigned with area unit weights.

In [2]:
set_params_columns(4)

In [3]:
%%render params
U_corridor = 35 / 1000 * ksf
U_floor = 35 / 1000 * ksf
U_roof = 25 / 1000 * ksf
U_wall = 12 / 1000 * ksf

<IPython.core.display.Latex object>

## Building Geometry

The lengths and widths of the weight areas are consistent across levels 2, 3, and 4.

In [4]:
set_params_columns(5)

In [5]:
%%render params
L_floor = 230 * ft
w_floor = (30 + 9 / 12) * ft
L_corridor = 230 * ft
w_corridor = 5.5 * ft
w_building = (w_floor * 2) + w_corridor

<IPython.core.display.Latex object>

The wall heights vary across the levels.

In [6]:
%%render params
h_wall__1 = 11.5 * ft
h_wall__2 = 10.5 * ft
h_wall__3 = 10.5 * ft
h_wall__4 = 10 * ft

<IPython.core.display.Latex object>

There is a small jog in the building floorplan. This does not affect the area, but the perimeter will be longer.

In [7]:
%%render params
L_jog = 5 * ft

<IPython.core.display.Latex object>

In [8]:
%%render
P_wall = (2 * L_floor) + (2 * w_building) + (2 * L_jog)

<IPython.core.display.Latex object>

Areas can now be calculated. Note that the areas of the walls are the actual wall areas; they have not been distributed to each floor based on tributary area, yet.

In [9]:
%%render 
A_floor = 2 * (L_floor * w_floor)
# This is a slight overestimate as the corridor doesn't stretch the full length of the building
A_corridor = L_corridor * w_corridor
A_level = L_floor * w_building
A_wall__1 = h_wall__1 * P_wall
A_wall__2 = h_wall__2 * P_wall
A_wall__3 = h_wall__3 * P_wall
A_wall__4 = h_wall__4 * P_wall

<IPython.core.display.Latex object>

## Component Weights

The typical floor and corridor weights are

In [10]:
%%render
W_floor__typ = U_floor * A_floor
W_corridor__typ = U_corridor * A_corridor

<IPython.core.display.Latex object>

This applies to levels 2, 3, and 4.

In [11]:
set_params_columns(3)

In [12]:
%%render params
W_floor__2 = W_floor__typ
W_floor__3 = W_floor__typ
W_floor__4 = W_floor__typ
W_corridor__2 = W_corridor__typ
W_corridor__3 = W_corridor__typ
W_corridor__4 = W_corridor__typ

<IPython.core.display.Latex object>

The roof weight is

In [13]:
%%render
W_roof = U_roof * A_level

<IPython.core.display.Latex object>

The exterior walls are divided in half to the floor above and below.

In [14]:
%%render
W_wall__2 = U_wall * (A_wall__2 / 2 + A_wall__1 / 2)
W_wall__3 = U_wall * (A_wall__3 / 2 + A_wall__2 / 2)
W_wall__4 = U_wall * (A_wall__4 / 2 + A_wall__3 / 2)
W_wall__roof = U_wall * (A_wall__4 / 2)

<IPython.core.display.Latex object>

## Level Weights

Each level's weight is

In [15]:
%%render 
W_level__2 = W_floor__2 + W_corridor__2 + W_wall__2
W_level__3 = W_floor__3 + W_corridor__3 + W_wall__3
W_level__4 = W_floor__4 + W_corridor__4 + W_wall__4
# be sure to add the weight of the truss later
W_level__roof = W_roof + W_wall__roof

<IPython.core.display.Latex object>

## Total Weight

The total weight of the structure is

In [16]:
%%render long
W_total = W_level__2 + W_level__3 + W_level__4 + W_level__roof

<IPython.core.display.Latex object>

# Seismic Equivalent Lateral Force (ELF) Analysis

We will be using the equivalent lateral force method prescribed by ASCE 7-16 Section 12.8. We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

## Seismic Parameters

In [17]:
set_params_columns(6)

In [18]:
S_S = 0.887  # g
S_1 = 0.410  # g
seismic_params = SeismicParameters(s_1=S_1, s_s=S_S, risk_category=RiskCategory.II, site_class=SiteClass.D)

Seismic design parameters are obtained from the geotech report and ASCE 7-16 Sec. 11 and 12.

In [19]:
%%render params
R = 6.5
Omega_0 = 3.0
C_d = 4.0
rho = 1.0

<IPython.core.display.Latex object>

In [20]:
set_params_columns(5)

In [21]:
%%render params
I_e = seismic_params.i_e
S_S = seismic_params.s_s  # g
S_1 = seismic_params.s_1  # g
F_a = seismic_params.f_a
F_v = seismic_params.f_v
S_MS = seismic_params.s_ms  # g
S_M1 = seismic_params.s_m1  # g
S_DS = seismic_params.s_ds  # g
S_D1 = seismic_params.s_d1  # g
SDC = seismic_params.sdc
C_u = 1.4
T_L = 16  # s

<IPython.core.display.Latex object>

## Base Shear

We must calculate the period

In [22]:
set_params_columns(1)

In [23]:
%%render params
h_top__roof = (57 + 8 / 12) * ft
h_bottom__roof = 45.5 * ft
h_n = np.mean([h_top__roof, h_bottom__roof])  # ASCE 7-16 Sec. 11.2

<IPython.core.display.Latex object>

In [24]:
%%render
h_n__check = check_value(h_n, 65 * ft, "leq")

<IPython.core.display.Latex object>

We will approximate the building period.

In [25]:
h_n = float(h_n)  # to prevent weirdness with units

In [26]:
%%render
C_t = 0.02  # ASCE 7-16 Table 12.8-2
x = 0.75  # ASCE 7-16 Table 12.8-2
T_a = C_t * h_n**x  # ASCE 7-16 Eq. 12.8-7

<IPython.core.display.Latex object>

Because we are not using an automated analysis software, we can make the following assumption:

In [27]:
%%render
T = T_a  # ASCE 7-16 Sec. 12.8.2

<IPython.core.display.Latex object>

With the period, we can calculate the seismic response coefficient.

In [28]:
%%render
C_s = S_DS / (R / I_e)  # ASCE 7-16 Eq. 12.8-2

<IPython.core.display.Latex object>

We must check that $C_s$ does not exceed:

In [29]:
%%render long
# fmt: off
if T <= T_L:  C_s__check__max = S_D1 / (T * (R / I_e));              C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
elif T > T_L: C_s__check__max = (S_D1 * T_L) / (T**2 * (R / I_e));   C_s__check__max = check_value(C_s, C_s__check__max, "leq")  # ASCE 7-16 Sec. 12.8.1.1
# fmt: on

<IPython.core.display.Latex object>

And we must also check that $C_s$ is not less than:

In [30]:
%%render
C_s__check__min__1 = 0.044 * S_DS * I_e  # ASCE 7-16 Eq. 12.8-5
C_s__check__min = check_value(C_s, C_s__check__min__1, "geq")

<IPython.core.display.Latex object>

as well as:

In [31]:
%%render
C_s__check__min = check_value(C_s, 0.01, "geq")  # ASCE 7-16 Eq. 12.8-5

<IPython.core.display.Latex object>

and finally:

In [32]:
%%render
# fmt: off
if S_1 >= 0.6:  C_s__check__min__3 = 0.5 * S_1 / (R / I_e); C_s__check__min = check_value(C_s, C_s__check__min__3, "geq")  # ASCE 7-16 Eq. 12.8-6
elif S_1 < 0.6: C_s__check__min = "\\text{skipping check}"  # ASCE 7-16 Eq. 12.8-6
# fmt: on

<IPython.core.display.Latex object>

And now, we can calculate the base shear, $V$

In [33]:
%%render
V = C_s * W_total  # ASCE 7-16 Eq. 12.8-1

<IPython.core.display.Latex object>

## Distribute the seismic forces

With the base shear calculated, we must now distribute the base shear on each level.

First, we must calculate the structural period exponent, $k$.

In [34]:
%%render
# fmt: off
if T <= 0.5: k = 1
elif T >= 2.5: k = 2
elif T > 0.5 and T < 2.5: k = 1 + ((T - 0.5) * (2 - 1)) / (2.5 - 0.5)
# fmt: on

<IPython.core.display.Latex object>

Now, we can distribute the forces per floor,

$$
F_x = C_{vx} V\;\;\text{(ASCE 7-16 Eq. 12.8-11)}
$$

As well as cumulatively from roof to ground.

$$
V_x = \sum_{i=x}^n F_i\;\; \text{(ASCE 7-16 Eq. 12.8-11)}
$$

In [35]:
# Load in basic data
seismic_loads = pd.DataFrame(
    {
        "Level": ["Ground", "Level 2", "Level 3", "Level 4", "Roof"],
        "floor height": [
            float(h_wall__1),
            float(h_wall__2),
            float(h_wall__3),
            float(h_wall__4),
            0,
        ],
        "floor weight": [
            0,
            float(W_level__2),
            float(W_level__3),
            float(W_level__4),
            float(W_level__roof),
        ],
    },
)
seismic_loads = seismic_loads.set_index("Level")

# Calc cumulative elevation
elevations = [0]
for height in seismic_loads["floor height"][:-1]:
    elevations.append(height + elevations[-1])
seismic_loads.insert(2, "elevation", elevations)

# Calc coefficients and story forces ASCE 7-16 Eq. 12.8-11 and 12.8-12
seismic_loads["wxhxk"] = seismic_loads["floor weight"] * seismic_loads["elevation"] ** float(k)
cumulative_wxhxk = seismic_loads["wxhxk"].sum()
seismic_loads["C_vx"] = seismic_loads["wxhxk"] / cumulative_wxhxk
seismic_loads["F_x"] = seismic_loads["C_vx"] * float(V)

# Calc story shear forces ASCE 7-16 Eq. 12.8-13
story_shears = [0]
for shear in reversed(list(seismic_loads["F_x"][1:])):
    story_shears.insert(0, shear + story_shears[0])
seismic_loads["V_x"] = story_shears

# Calc overturning moments
floor_moments = seismic_loads["V_x"] * seismic_loads["floor height"]
overturning_moments = [0]
for floor_moment in reversed(list(floor_moments[:-1])):
    overturning_moments.insert(0, overturning_moments[0] + floor_moment)
seismic_loads["OTM"] = overturning_moments

In [36]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "floor height": "$h_{floor}$ [ft]",
    "floor weight": "$w_x$ [kip]",
    "elevation": "$h_x$ [ft]",
    "wxhxk": "$w_x h_x^k$",
    "C_vx": "$C_{vx}$",
    "F_x": "$F_x$ [kip]",
    "V_x": "$V_x$ [kip]",
    "OTM": "OTM [kip-ft]",
}

Calculating for each floor leads to the following \autoref{tab:seismic_forces_calculation_table}

In [37]:
# Format the dataframe
# Reverse the order of the dataframe with '.loc[::-1]' so roof is the top row and ground is the bottom row, then apply formatting with the 'format_dataframe' function
display_table(
    dataframe=seismic_loads.iloc[::-1],
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Seismic Forces Calculation Table",
    label="tab:seismic_forces_calculation_table",
)

,$h_{floor}$ [ft],$w_x$ [kip],$h_x$ [ft],$w_x h_x^k$,$C_{vx}$,$F_x$ [kip],$V_x$ [kip],OTM [kip-ft]
Level,,,,,,,,
Roof,0.0,421.490,42.5,17913.325,0.306127,72.387392,0.000000,0.000000
Level 4,10.0,613.642,32.5,19943.365,0.340819,80.590744,72.387392,723.873920
Level 3,10.5,615.454,22.0,13539.988,0.231389,54.714824,152.978136,2330.144343
Level 2,10.5,619.078,11.5,7119.397,0.121666,28.769342,207.692959,4510.920416
Ground,11.5,0.000,0.0,0.000,0.000000,0.000000,236.462302,7230.236885


# Diaphragm and Shear Wall Design

## Redundancy Factor

The redundancy factor, $\rho$, is either 1.3 or 1.0 according to ASCE 7-16 12.3.4. \

No single story resists more than 35% of the shear force (see $C_{vx}$ in \autoref{tab:seismic_forces_calculation_table}), so all values of $\rho$ are $1.0$.

Table: Redundancy Factors {#tab:redundancy_factors}

## Envelope Procedure

We will envelope the forces with a fully flexible diaphragm assumption and a fully rigid diaphragm assumption.

In [38]:
# Set up diaphragm dataframe
diaphragm_rigidity = ["Flexible", "Rigid"]
levels = ["Roof", "Level 4", "Level 3", "Level 2"]
walls = {
    "A-2": "N-S",
    "A-2.4": "N-S",
    "A-3.7": "N-S",
    "A-4": "N-S",
    "A-5": "N-S",
    "A-5.1": "N-S",
    "A-6": "N-S",
    "A-6.1": "N-S",
    "A-6.8": "N-S",
    "A-7": "N-S",
    "A-7.8": "N-S",
    "A-8": "N-S",
    "A-8.2": "N-S",
    "A-8.9": "N-S",
    "A-9": "N-S",
    # E-W
    "A-B.1": "E-W",
    "A-B.2.5": "E-W",
    "A-B.3.5": "E-W",
    "A-B.4": "E-W",
    "A-B.4.8": "E-W",
    "A-B.5.5": "E-W",
    "A-B.6.5": "E-W",
    "A-B.7.9": "E-W",
    "A-B.8.8": "E-W",
    "A-C.1.8": "E-W",
    "A-C.3.2": "E-W",
    "A-C.4.2": "E-W",
    "A-C.5.5": "E-W",
    "A-C.6.5": "E-W",
    "A-C.6.9": "E-W",
    "A-C.7.5": "E-W",
    "A-C.8.5": "E-W",
    "A-C.9.2": "E-W",
}

index = pd.MultiIndex.from_product(
    [diaphragm_rigidity, levels, walls.keys()],
    names=["Diaphragm Rigidity", "Level", "Wall"],
)

shear_walls = pd.DataFrame(index=index)
shear_walls["Direction"] = shear_walls.index.get_level_values("Wall").map(walls)
flexible = shear_walls.loc["Flexible"].copy()
flexible_ns = flexible[flexible["Direction"] == "N-S"].copy()
flexible_ew = flexible[flexible["Direction"] == "E-W"].copy()
rigid = shear_walls.loc["Rigid"].copy()
rigid_ns = rigid[rigid["Direction"] == "N-S"].copy()
rigid_ew = rigid[rigid["Direction"] == "E-W"].copy()

### Flexible Diaphragm Assumption

We will uniformly distribute the shear forces across the diaphragms based on tributary areas of each level. This removes some nuance in the distribution of floor weight distribution (corridors are 17% heavier than the rest of the floors, for instance), but the vast majority of the area of each level is typical floor weight (30 psf) instead of corridor weight (35 psf).

$$
f_{p,x} = F_x / A_{level}
$$

In [39]:
%%render
A_level
f_p__roof = seismic_loads.loc["Roof", "F_x"] * 1000 * lb / A_level
f_p__4 = seismic_loads.loc["Level 4", "F_x"] * 1000 * lb / A_level
f_p__3 = seismic_loads.loc["Level 3", "F_x"] * 1000 * lb / A_level
f_p__2 = seismic_loads.loc["Level 2", "F_x"] * 1000 * lb / A_level

<IPython.core.display.Latex object>

#### North-South Direction

Most of the shear walls are not in line which presents an issue: if we assume a diaphragm is the full depth, there will be an extremely long collector element without a shear wall on both sides. We will avoid this by just assuming the flexible diaphragm does not extend the full depth and instead is just tributary to the nearest wall (\autoref{fig:diaphragm_trib_area}). This means that there are essentially two diaphragms through the depth of the building. This is a rough approximation to ensure all the shear load is resolved in the shear walls. \

The walls that are close to inline (A-5 and A-5.1, A-6 and A-6.1, and A-9 and A-9.1) in reality will be treated as inline and have collectors to transfer the shear load from the diaphragm to those walls.

Proper detailing will be done to ensure continuity through the depth of the diaphragm.

![Shear walls (thick red lines) in plan view with their (colored) tributary areas.](images/shear_wall_tributary_area.png){width=60% #fig:diaphragm_trib_area}

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ns}, \autoref{tab:shear_wall_forces_level4_ns}, \autoref{tab:shear_wall_forces_level3_ns}, and \autoref{tab:shear_wall_forces_level2_ns}.

Shear wall sheathing will be assumed to have the following properties:

1. WSP Sheathing
2. 15/32" thickness
3. 10d common nails
4. Plywood panels
5. Framing uses Douglas-Fir-Larch (no specific gravity adjustment)

Notes: 

- Cumulative shear force, $F_{x,cum}$, is cumulative from the Roof downwards. For example, Level 3 includes the shear forces from the Roof, Level 4, and Level 3.

In [40]:
from structural_tools.seismic.timber.sheathing import (
    DesignMethod,
    LoadCase,
    Nail,
    PanelType,
    Sheathing,
    SheathingMaterial,
    get_sheathing_properties,
)

# Set baseline sheathing properties. We set the number of sheathed sides later
sheathing = Sheathing(
    sheathing_material=SheathingMaterial.WSP_SHEATHING,
    minimum_nominal_panel_thickness=15 / 32,
    nail=Nail.COMMON_10D,
    panel_type=PanelType.PLY,
)

In [41]:
# Plan dimensions
l_A1__A2 = (31 + 11 / 12) * ft
l_A2__A3 = (17 + 11 / 12) * ft
l_A3__A4 = (26 + 4 / 12) * ft
l_A4__A5 = (21 + 4 / 12) * ft
l_A5__A6 = (24 + 8 / 12) * ft
l_A6__A7 = (34 + 4 / 12) * ft
l_A7__A8 = (26 + 1 / 12) * ft
l_A8__A9 = (21 + 4 / 12) * ft
l_A9__A10 = (31 + 1 / 12) * ft

l_A1__A24 = (40 + 1 / 12) * ft
l_A24__A35 = (23 + 8 / 12) * ft
l_A35__A51 = (34 + 4 / 12) * ft
l_A51__A61 = (24 + 8 / 12) * ft
l_A61__A68 = (24 + 8 / 12) * ft
l_A68__A82 = (38 + 3 / 12) * ft
l_A82__A89 = (17 + 11 / 12) * ft
l_A89__A10 = (31 + 5 / 12) * ft

l_A37__A4 = (8 + 10 / 12) * ft
l_A3__A37 = l_A3__A4 - l_A37__A4
l_A78__A82 = (8 + 10 / 12) * ft
l_A68__A78 = l_A68__A82 - l_A78__A82

d_trib_ns = 67 / 2 * ft

In [42]:
# Wall tributary widths and lengths
t_w__A2 = l_A1__A2 + (l_A2__A3 + l_A3__A37) / 2
t_w__A24 = (l_A1__A24 - L_jog) + ((l_A24__A35 + l_A35__A51) / 2)
t_w__A37 = ((l_A2__A3 + l_A3__A37) / 2) + (l_A37__A4 / 2)
t_w__A4 = (l_A37__A4 / 2) + (l_A4__A5 / 2)
t_w__A5 = (l_A4__A5 / 2) + (l_A5__A6 / 2)
t_w__A51 = ((l_A24__A35 + l_A35__A51) / 2) + (l_A51__A61 / 2)
t_w__A6 = (l_A5__A6 / 2) + (l_A6__A7 / 2)
t_w__A61 = (l_A51__A61 / 2) + (l_A61__A68 / 2)
t_w__A68 = (l_A61__A68 / 2) + (l_A68__A78 / 2)
t_w__A7 = (l_A6__A7 / 2) + (l_A7__A8 / 2)
t_w__A78 = (l_A68__A78 / 2) + (l_A78__A82 / 2)
t_w__A8 = (l_A7__A8 / 2) + (l_A8__A9 / 2)
t_w__A82 = (l_A78__A82 / 2) + (l_A82__A89 / 2)
t_w__A89 = (l_A82__A89 / 2) + l_A89__A10
t_w__A9 = (l_A8__A9 / 2) + (l_A9__A10 - L_jog)

t_l__A2 = float(d_trib_ns)
t_l__A24 = float(d_trib_ns)
t_l__A37 = float(d_trib_ns)
t_l__A4 = float(d_trib_ns)
t_l__A5 = float(d_trib_ns)
t_l__A51 = float(d_trib_ns)
t_l__A6 = float(d_trib_ns)
t_l__A61 = float(d_trib_ns)
t_l__A68 = float(d_trib_ns)
t_l__A7 = float(d_trib_ns)
t_l__A78 = float(d_trib_ns)
t_l__A8 = float(d_trib_ns)
t_l__A82 = float(d_trib_ns)
t_l__A89 = float(d_trib_ns)
t_l__A9 = float(d_trib_ns)

In [43]:
flexible_ns.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ns.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ns.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ns.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ns.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ns.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ns.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ns.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
l_aa__ab = 30 + 9 / 12
l_ac__ad = 30 + 9 / 12
l_stairwall = 20
wall_tributary = {
    "A-2": {"trib_width": float(t_w__A2), "trib_depth": t_l__A2, "wall_length": l_aa__ab},
    "A-2.4": {"trib_width": float(t_w__A24), "trib_depth": t_l__A24, "wall_length": l_ac__ad},
    "A-3.7": {"trib_width": float(t_w__A37), "trib_depth": t_l__A37, "wall_length": l_stairwall},
    "A-4": {"trib_width": float(t_w__A4), "trib_depth": t_l__A4, "wall_length": l_stairwall},
    "A-5": {"trib_width": float(t_w__A5), "trib_depth": t_l__A5, "wall_length": l_aa__ab},
    "A-5.1": {"trib_width": float(t_w__A51), "trib_depth": t_l__A51, "wall_length": l_ac__ad},
    "A-6": {"trib_width": float(t_w__A6), "trib_depth": t_l__A6, "wall_length": l_aa__ab},
    "A-6.1": {"trib_width": float(t_w__A61), "trib_depth": t_l__A61, "wall_length": l_ac__ad},
    "A-6.8": {"trib_width": float(t_w__A68), "trib_depth": t_l__A68, "wall_length": l_ac__ad},
    "A-7": {"trib_width": float(t_w__A7), "trib_depth": t_l__A7, "wall_length": l_aa__ab},
    "A-7.8": {"trib_width": float(t_w__A78), "trib_depth": t_l__A78, "wall_length": l_stairwall},
    "A-8": {"trib_width": float(t_w__A8), "trib_depth": t_l__A8, "wall_length": l_aa__ab},
    "A-8.2": {"trib_width": float(t_w__A82), "trib_depth": t_l__A82, "wall_length": l_stairwall},
    "A-8.9": {"trib_width": float(t_w__A89), "trib_depth": t_l__A89, "wall_length": l_ac__ad},
    "A-9": {"trib_width": float(t_w__A9), "trib_depth": t_l__A9, "wall_length": l_aa__ab},
}

flexible_ns["wall length"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary width"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = flexible_ns.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ns["tributary depth"] = float(d_trib_ns)
flexible_ns["tributary area"] = flexible_ns["tributary depth"] * flexible_ns["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ns["tributary area"].sum() / 4
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up. Ensure you are including all areas in your calculations"
)

flexible_ns["floor shear"] = flexible_ns["tributary area"] * flexible_ns["tributary weight"]
flexible_ns["cumulative shear"] = flexible_ns.groupby(level="Wall")["floor shear"].cumsum()
flexible_ns["unit shear"] = flexible_ns["cumulative shear"] / flexible_ns["wall length"]
asd_load_factor = 0.7
flexible_ns["adjusted shear demand"] = flexible_ns["unit shear"] * asd_load_factor
flexible_ns["sheathed sides"] = 1
flexible_ns["aspect ratio"] = flexible_ns["wall height"] / flexible_ns["wall length"]

#### Check aspect ratios

We must check the aspect ratios of the shear walls to ensure we don't need to apply the aspect ratio factor in SDPWS Sec. 4.3.3.2. The maximum aspect ratio of all N-S walls is:

In [44]:
%%render
ratio_max = check_value(flexible_ns["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

In [71]:
if "OK" in ratio_max:
    display_text(r"$\therefore$ All aspect ratios are OK")
else:
    display_text("The aspect ratio knockdown factor must be applied.")

'$\\therefore$ All aspect ratios are OK'

In [45]:
flexible_ns[["adjusted shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = get_sheathing_properties(
    flexible_ns[["adjusted shear demand", "sheathed sides"]], sheathing=sheathing
)

shear_dcr_check_value = 1

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ns["shear dcr"] > shear_dcr_check_value).any():
    flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ns.loc[
        flexible_ns["shear dcr"] > shear_dcr_check_value,
        ["adjusted shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ns.loc[flexible_ns["shear dcr"] > shear_dcr_check_value, ["adjusted shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )

In [46]:
# Update dataframe
flexible = flexible.reindex(columns=flexible_ns.columns)
flexible.update(flexible_ns)

In [47]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$w_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "floor shear": "$F_x$ [lbf]",
    "cumulative shear": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear": "$v$ [plf]",
    "adjusted shear demand": "$0.7 v$ [lbf]",
    "adjusted shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [48]:
def highlight_insufficient_capacity(row):
    styles = [""] * len(row)

    if row["$0.7 v$ [plf]"] > row["$v_{cap}$ [plf]"]:
        demand_idx = row.index.get_loc("$0.7 v$ [plf]")
        capacity_idx = row.index.get_loc("$v_{cap}$ [plf]")

        styles[demand_idx] = "background-color: red; font-weight: bold"
        styles[capacity_idx] = "background-color: red; font-weight: bold"

    return styles

In [49]:
display_table(
    dataframe=flexible_ns,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Roof",
    label=f"tab:shear_wall_forces_roof_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 4",
    label=f"tab:shear_wall_forces_level4_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 3",
    label=f"tab:shear_wall_forces_level3_ns",
)
display_table(
    dataframe=flexible_ns,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces NS - Level 2",
    label=f"tab:shear_wall_forces_level2_ns",
)

,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,4.69743,1662.437500,7809.183325,7809.183325,30.75,253.957181,177.770027,310.714286,1,6.0
A-2.4,4.69743,2146.791667,10084.402984,10084.402984,30.75,327.948065,229.563645,310.714286,1,6.0
A-3.7,4.69743,741.187500,3481.676193,3481.676193,20.00,174.083810,121.858667,310.714286,1,6.0
A-4,4.69743,505.291667,2373.572094,2373.572094,20.00,118.678605,83.075023,310.714286,1,6.0
A-5,4.69743,770.500000,3619.369601,3619.369601,30.75,117.703076,82.392154,310.714286,1,6.0
A-5.1,4.69743,1384.666667,6504.374356,6504.374356,30.75,211.524369,148.067059,310.714286,1,6.0
A-6,4.69743,988.250000,4642.234923,4642.234923,30.75,150.966989,105.676893,310.714286,1,6.0
A-6.1,4.69743,826.333333,3881.642761,3881.642761,30.75,126.232285,88.362599,310.714286,1,6.0
A-6.8,4.69743,905.895833,4255.382013,4255.382013,30.75,138.386407,96.870485,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,5.229769,1662.437500,8694.164450,16503.347775,30.75,536.694237,375.685966,460.714286,1,4.0
A-2.4,5.229769,2146.791667,11227.224957,21311.627941,30.75,693.061071,485.142750,600.000000,1,3.0
A-3.7,5.229769,741.187500,3876.239566,7357.915758,20.00,367.895788,257.527052,310.714286,1,6.0
A-4,5.229769,505.291667,2642.558800,5016.130894,20.00,250.806545,175.564581,310.714286,1,6.0
A-5,5.229769,770.500000,4029.537176,7648.906777,30.75,248.744936,174.121455,310.714286,1,6.0
A-5.1,5.229769,1384.666667,7241.487099,13745.861455,30.75,447.019885,312.913919,460.714286,1,4.0
A-6,5.229769,988.250000,5168.319421,9810.554345,30.75,319.042418,223.329692,310.714286,1,6.0
A-6.1,5.229769,826.333333,4321.532623,8203.175384,30.75,266.769931,186.738952,310.714286,1,6.0
A-6.8,5.229769,905.895833,4737.626136,8993.008149,30.75,292.455550,204.718885,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,3.550605,1662.437500,5902.658971,22406.006745,30.75,728.650626,510.055438,600.000000,1,3.0
A-2.4,3.550605,2146.791667,7622.409317,28934.037258,30.75,940.944301,658.661011,769.642857,1,2.0
A-3.7,3.550605,741.187500,2631.664075,9989.579834,20.00,499.478992,349.635294,460.714286,1,4.0
A-4,3.550605,505.291667,1794.091140,6810.222033,20.00,340.511102,238.357771,310.714286,1,6.0
A-5,3.550605,770.500000,2735.741185,10384.647963,30.75,337.712129,236.398490,310.714286,1,6.0
A-5.1,3.550605,1384.666667,4916.404449,18662.265904,30.75,606.902956,424.832069,460.714286,1,4.0
A-6,3.550605,988.250000,3508.885433,13319.439778,30.75,433.152513,303.206759,310.714286,1,6.0
A-6.1,3.550605,826.333333,2933.983300,11137.158685,30.75,362.184022,253.528816,310.714286,1,6.0
A-6.8,3.550605,905.895833,3216.478314,12209.486463,30.75,397.056470,277.939529,310.714286,1,6.0


,$w_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [lbf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-2,1.866927,1662.437500,3103.649174,25509.655919,30.75,829.582306,580.707614,600.000000,1,3.0
A-2.4,1.866927,2146.791667,4007.902963,32941.940221,30.75,1071.282609,749.897826,769.642857,1,2.0
A-3.7,1.866927,741.187500,1383.742831,11373.322664,20.00,568.666133,398.066293,460.714286,1,4.0
A-4,1.866927,505.291667,943.342570,7753.564603,20.00,387.678230,271.374761,310.714286,1,6.0
A-5,1.866927,770.500000,1438.467123,11823.115086,30.75,384.491548,269.144083,310.714286,1,6.0
A-5.1,1.866927,1384.666667,2585.071352,21247.337256,30.75,690.970317,483.679222,600.000000,1,3.0
A-6,1.866927,988.250000,1844.990441,15164.430219,30.75,493.152202,345.206542,460.714286,1,4.0
A-6.1,1.866927,826.333333,1542.703871,12679.862556,30.75,412.353254,288.647278,310.714286,1,6.0
A-6.8,1.866927,905.895833,1691.241237,13900.727701,30.75,452.056185,316.439330,460.714286,1,4.0


#### East-West Direction

Most likely, the shear walls in the E-W direction will be governed by a rigid diaphragm assumption. 

Shear wall shear forces are tabulated in \autoref{tab:shear_wall_forces_roof_ew}, \autoref{tab:shear_wall_forces_level4_ew}, \autoref{tab:shear_wall_forces_level3_ew}, and \autoref{tab:shear_wall_forces_level2_ew}. \

In [50]:
# Plan dimensions
l_AA__AB = (30 + 9 / 12) * ft
l_AB__AC = (5 + 6 / 12) * ft
l_AC__AD = (30 + 9 / 12) * ft

d_trib_ew = L_floor

In [51]:
# Wall tributary widths and lengths
t_w__AB = l_AA__AB + l_AB__AC / 2
t_w__AC = l_AB__AC / 2 + l_AC__AD

t_l__AB__1 = 33.75
t_l__AB__2 = 22.25
t_l__AB__3 = 16
t_l__AB__4 = 19
t_l__AB__5 = 16
t_l__AB__6 = 34.5
t_l__AB__7 = 28.25
t_l__AB__8 = 28.5
t_l__AB__9 = 31.75

t_l__AC__1 = 44
t_l__AC__2 = 31
t_l__AC__3 = 27.5
t_l__AC__4 = 31
t_l__AC__5 = 15.5
t_l__AC__6 = 12.34
t_l__AC__7 = 19
t_l__AC__8 = 21.33
t_l__AC__9 = 28.33

In [52]:
# Wall segment lengths
l_AB__1 = 21
l_AB__2 = 13.25
l_AB__3 = 9
l_AB__4 = 12
l_AB__5 = 11
l_AB__6 = 29.75
l_AB__7 = 12.75
l_AB__8 = 16.25
l_AB__9 = 13.75

l_AC__1 = 17.25
l_AC__2 = 23
l_AC__3 = 22.5
l_AC__4 = 26
l_AC__5 = 11.25
l_AC__6 = 7
l_AC__7 = 6.75
l_AC__8 = 10.25
l_AC__9 = 25.25

In [53]:
flexible_ew.loc["Roof", "tributary weight"] = float(f_p__roof)
flexible_ew.loc["Level 4", "tributary weight"] = float(f_p__4)
flexible_ew.loc["Level 3", "tributary weight"] = float(f_p__3)
flexible_ew.loc["Level 2", "tributary weight"] = float(f_p__2)

# Shear wall height is for the floor below
flexible_ew.loc["Roof", "wall height"] = float(seismic_loads.loc["Level 4", "floor height"])
flexible_ew.loc["Level 4", "wall height"] = float(seismic_loads.loc["Level 3", "floor height"])
flexible_ew.loc["Level 3", "wall height"] = float(seismic_loads.loc["Level 2", "floor height"])
flexible_ew.loc["Level 2", "wall height"] = float(seismic_loads.loc["Ground", "floor height"])

# Add wall tributary widths and wall lengths to df
wall_tributary = {
    "A-B.1": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__1, "wall_length": l_AB__1},
    "A-B.2.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__2, "wall_length": l_AB__2},
    "A-B.3.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__3, "wall_length": l_AB__3},
    "A-B.4": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__4, "wall_length": l_AB__4},
    "A-B.4.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__5, "wall_length": l_AB__5},
    "A-B.5.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__6, "wall_length": l_AB__6},
    "A-B.6.5": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__7, "wall_length": l_AB__7},
    "A-B.7.9": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__8, "wall_length": l_AB__8},
    "A-B.8.8": {"trib_width": float(t_w__AB), "trib_depth": t_l__AB__9, "wall_length": l_AB__9},
    "A-C.1.8": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__1, "wall_length": l_AC__1},
    "A-C.3.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__2, "wall_length": l_AC__2},
    "A-C.4.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__3, "wall_length": l_AC__3},
    "A-C.5.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__4, "wall_length": l_AC__4},
    "A-C.6.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__5, "wall_length": l_AC__5},
    "A-C.6.9": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__6, "wall_length": l_AC__6},
    "A-C.7.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__7, "wall_length": l_AC__7},
    "A-C.8.5": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__8, "wall_length": l_AC__8},
    "A-C.9.2": {"trib_width": float(t_w__AC), "trib_depth": t_l__AC__9, "wall_length": l_AC__9},
}

flexible_ew["wall length"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["wall_length"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary width"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_width"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary depth"] = flexible_ew.index.get_level_values("Wall").map(
    {key: value["trib_depth"] for key, value in wall_tributary.items()}
)
flexible_ew["tributary area"] = flexible_ew["tributary depth"] * flexible_ew["tributary width"]

# Ensure that the calculated areas from earlier and tributary areas match
trib_area = flexible_ew["tributary area"].sum() / 4
assert np.isclose(trib_area, float(A_level)), (
    f"Shear wall tributary areas ({trib_area}) and level area ({float(A_level)}) do not match up.\n\
    Ensure you are including all areas in your calculations"
)

flexible_ew["floor shear"] = flexible_ew["tributary area"] * flexible_ew["tributary weight"]
flexible_ew["cumulative shear"] = flexible_ew.groupby(level="Wall")["floor shear"].cumsum()
flexible_ew["unit shear"] = flexible_ew["cumulative shear"] / flexible_ew["wall length"]
asd_load_factor = 0.7
flexible_ew["adjusted shear demand"] = flexible_ew["unit shear"] * asd_load_factor
flexible_ew["sheathed sides"] = 1
flexible_ew["aspect ratio"] = flexible_ew["wall height"] / flexible_ew["wall length"]

#### Check aspect ratios

We will also check the aspect ratios of the E-W walls.

In [54]:
%%render
ratio_max = check_value(flexible_ew["aspect ratio"].max(), 2, "<")

<IPython.core.display.Latex object>

$\therefore$ All aspect ratios are OK

In [55]:
# using sheathing from the NS calc in memory

flexible_ew[["adjusted shear capacity", "nail spacing", "shear stiffness", "shear dcr"]] = get_sheathing_properties(
    flexible_ew[["adjusted shear demand", "sheathed sides"]],
    sheathing=sheathing,
)

# Assign 2-sided sheathing to walls requiring it, then recalc capacity for just those walls (more efficient than passing the full wall dataframe)
if (flexible_ew["shear dcr"] > shear_dcr_check_value).any():
    flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, "sheathed sides"] = 2
    flexible_ew.loc[
        flexible_ew["shear dcr"] > shear_dcr_check_value,
        ["adjusted shear capacity", "nail spacing", "shear stiffness", "shear dcr"],
    ] = get_sheathing_properties(
        flexible_ew.loc[flexible_ew["shear dcr"] > shear_dcr_check_value, ["adjusted shear demand", "sheathed sides"]],
        sheathing=sheathing,
    )

In [56]:
# Update dataframe
flexible.update(flexible_ew)

In [57]:
# Rename columns for display with LaTeX formatting for units and subscripts/superscripts
column_name_map = {
    "tributary weight": "$W_{trib}$ [psf]",
    "tributary area": "$A$ [ft$^2$]",
    "floor shear": "$F_x$ [lbf]",
    "cumulative shear": "$F_{x,cum}$ [lbf]",
    "wall length": "$l_{wall}$ [ft]",
    "unit shear": "$v$ [plf]",
    "adjusted shear demand": "$0.7 v$ [plf]",
    "adjusted shear capacity": "$v_{cap}$ [plf]",
    "sheathed sides": "sides",
    "nail spacing": "$s_{nail}$ [in]",
}

In [58]:
display_table(
    dataframe=flexible_ew,
    levels="Roof",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Roof",
    label=f"tab:shear_wall_forces_roof_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 4",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 4",
    label=f"tab:shear_wall_forces_level4_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 3",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 3",
    label=f"tab:shear_wall_forces_level3_ew",
)
display_table(
    dataframe=flexible_ew,
    levels="Level 2",
    column_names_filter_and_map=column_name_map,
    position_float="centering",
    caption="Shear wall forces EW - Level 2",
    label=f"tab:shear_wall_forces_level2_ew",
)

,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B.1,4.69743,1130.625,5311.031480,5311.031480,21.00,252.906261,177.034383,310.714286,1,6.0
A-B.2.5,4.69743,745.375,3501.346680,3501.346680,13.25,264.252580,184.976806,310.714286,1,6.0
A-B.3.5,4.69743,536.000,2517.822331,2517.822331,9.00,279.758037,195.830626,310.714286,1,6.0
A-B.4,4.69743,636.500,2989.914019,2989.914019,12.00,249.159502,174.411651,310.714286,1,6.0
A-B.4.8,4.69743,536.000,2517.822331,2517.822331,11.00,228.892939,160.225057,310.714286,1,6.0
A-B.5.5,4.69743,1155.750,5429.054402,5429.054402,29.75,182.489224,127.742457,310.714286,1,6.0
A-B.6.5,4.69743,946.375,4445.530054,4445.530054,12.75,348.669024,244.068317,310.714286,1,6.0
A-B.7.9,4.69743,954.750,4484.871028,4484.871028,16.25,275.992063,193.194444,310.714286,1,6.0
A-B.8.8,4.69743,1063.625,4996.303689,4996.303689,13.75,363.367541,254.357279,310.714286,1,6.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B.1,5.229769,1130.625,5912.907812,11223.939293,21.00,534.473300,374.131310,460.714286,1,4.0
A-B.2.5,5.229769,745.375,3898.139224,7399.485904,13.25,558.451766,390.916236,460.714286,1,4.0
A-B.3.5,5.229769,536.000,2803.156296,5320.978628,9.00,591.219848,413.853893,460.714286,1,4.0
A-B.4,5.229769,636.500,3328.748102,6318.662120,12.00,526.555177,368.588624,460.714286,1,4.0
A-B.4.8,5.229769,536.000,2803.156296,5320.978628,11.00,483.725330,338.607731,460.714286,1,4.0
A-B.5.5,5.229769,1155.750,6044.305764,11473.360166,29.75,385.659165,269.961416,310.714286,1,6.0
A-B.6.5,5.229769,946.375,4949.322836,9394.852889,12.75,736.851207,515.795845,600.000000,1,3.0
A-B.7.9,5.229769,954.750,4993.122153,9477.993180,16.25,583.261119,408.282783,460.714286,1,4.0
A-B.8.8,5.229769,1063.625,5562.513275,10558.816964,13.75,767.913961,537.539773,600.000000,1,3.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B.1,3.550605,1130.625,4014.402826,15238.342119,21.00,725.635339,507.944737,600.000000,1,3.0
A-B.2.5,3.550605,745.375,2646.532234,10046.018138,13.25,758.190048,530.733034,600.000000,1,3.0
A-B.3.5,3.550605,536.000,1903.124303,7224.102931,9.00,802.678103,561.874672,600.000000,1,3.0
A-B.4,3.550605,636.500,2259.960110,8578.622230,12.00,714.885186,500.419630,600.000000,1,3.0
A-B.4.8,3.550605,536.000,1903.124303,7224.102931,11.00,656.736630,459.715641,460.714286,1,4.0
A-B.5.5,3.550605,1155.750,4103.611778,15576.971944,29.75,523.595696,366.516987,460.714286,1,4.0
A-B.6.5,3.550605,946.375,3360.203847,12755.056737,12.75,1000.396607,700.277625,769.642857,1,2.0
A-B.7.9,3.550605,954.750,3389.940165,12867.933345,16.25,791.872821,554.310975,600.000000,1,3.0
A-B.8.8,3.550605,1063.625,3776.512289,14335.329253,13.75,1042.569400,729.798580,769.642857,1,2.0


,$W_{trib}$ [psf],$A$ [ft$^2$],$F_x$ [lbf],"$F_{x,cum}$ [lbf]",$l_{wall}$ [ft],$v$ [plf],$0.7 v$ [plf],$v_{cap}$ [plf],sides,$s_{nail}$ [in]
Wall,,,,,,,,,,
A-B.1,1.866927,1130.625,2110.794148,17349.136267,21.00,826.149346,578.304542,600.000000,1,3.0
A-B.2.5,1.866927,745.375,1391.560587,11437.578724,13.25,863.213489,604.249442,769.642857,1,2.0
A-B.3.5,1.866927,536.000,1000.672781,8224.775712,9.00,913.863968,639.704778,769.642857,1,2.0
A-B.4,1.866927,636.500,1188.298928,9766.921158,12.00,813.910096,569.737068,600.000000,1,3.0
A-B.4.8,1.866927,536.000,1000.672781,8224.775712,11.00,747.706883,523.394818,600.000000,1,3.0
A-B.5.5,1.866927,1155.750,2157.700685,17734.672629,29.75,596.123450,417.286415,460.714286,1,4.0
A-B.6.5,1.866927,946.375,1766.812880,14521.869616,12.75,1138.970166,797.279116,921.428571,2,4.0
A-B.7.9,1.866927,954.750,1782.448392,14650.381737,16.25,901.561953,631.093367,769.642857,1,2.0
A-B.8.8,1.866927,1063.625,1985.710051,16321.039303,13.75,1186.984677,830.889274,921.428571,2,4.0


### Rigid Diaphragm Assumption

With a rigid diaphragm assumption, we now must calculate the following in each direction:

1. The relative stiffnesses of the shear walls to obtain a center of rigidity.
2. The center of mass of the diaphragm with the center of rigidity to obtain an eccentricity.

The relative stiffnesses of the shear walls can be calculated by using the deflection equation from NDS SDPWS Sec. 4.3.4

\begin{align*}
k &= \frac{F}{\delta},\, \text{assume }F = \text{induced load demand in the shear wall} \\
\delta_{sw} &= \frac{8vh^3}{EAb} + \frac{vh}{1000 G_a} + \frac{h \Delta_A}{b}\quad \text{(SDPWS Eq. 4.3-1)}
\end{align*}

where

- $b =$ the shear wall length [ft]
- $\Delta_A =$ vertical deformation of the wall overturning anchorage system + vertical compression deformation [in]
- $E =$ modulus of elasticity of end posts [psi]
- $A =$ area of end post cross-section [in$^2$]
- $G_a =$ apparent shear wall shear stiffness from nail slip and panel shear deformation [kip/in]
- $h =$ shear wall height [ft]
- $v =$ unit shear force induced by the design load [plf]
- $\delta_{sw} =$ maximum shear wall deflection determined by elastic analysis [in]

Note that for the following calculations, we will conservatively use the unit shear force induced by the nominal load, not the design load, for the value of $v$. This is because the stiffnesses obtained from this use will allow us to use the stiffnesses for deflection and story drift calculations

We will assume that the end posts are (2) 2x4 Douglas Fir No. 2; this means:

In [59]:
%%render
E_endpost = 1_600_000 * psi
A_endpost = 2 * 5.25 * inch**2

<IPython.core.display.Latex object>

We will assume a continuous tiedown rod system. A conservative first-pass value for $\Delta_A$ can be:

In [60]:
%%render
Delta_A = 0.25 * inch

<IPython.core.display.Latex object>

#### North-South Direction

In [61]:
rigid_ns["unit shear"] = flexible_ns["unit shear"]
rigid_ns["wall length"] = flexible_ns["wall length"]
rigid_ns["wall height"] = flexible_ns["wall height"]
rigid_ns["shear stiffness"] = flexible_ns["shear stiffness"]
rigid_ns["nail spacing"] = flexible_ns["nail spacing"]
rigid_ns["end post youngs modulus"] = float(E_endpost)
rigid_ns["end post area"] = float(A_endpost)
rigid_ns["Delta_A"] = float(Delta_A)

rigid_ns["delta_sw"] = (
    8
    * rigid_ns["unit shear"]
    * rigid_ns["wall height"] ** 3
    / (rigid_ns["end post youngs modulus"] * rigid_ns["end post area"] * rigid_ns["wall length"])
    + rigid_ns["unit shear"] * rigid_ns["wall height"] / (1000 * rigid_ns["shear stiffness"])
    + rigid_ns["wall height"] * rigid_ns["Delta_A"] / rigid_ns["wall length"]
)

##### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

$$
k = \frac{F_x}{\delta_{sw}}
$$

In [62]:
rigid_ns["cumulative shear"] = flexible_ns["cumulative shear"]
rigid_ns["wall stiffness"] = rigid_ns["cumulative shear"] / rigid_ns["delta_sw"]
rigid_ns["total level stiffness"] = rigid_ns.groupby("Level")["wall stiffness"].transform("sum")
rigid_ns["total level cumulative shear"] = rigid_ns.groupby("Level")["cumulative shear"].transform("sum")
rigid_ns["relative wall stiffness"] = rigid_ns["wall stiffness"] / rigid_ns["total level stiffness"]
rigid_ns["direct shear force"] = rigid_ns["total level cumulative shear"] * rigid_ns["relative wall stiffness"]
rigid_ns

Direction   unit shear  wall length  wall height  \
Level   Wall                                                     
Roof    A-2         N-S   253.957181        30.75         10.0   
        A-2.4       N-S   327.948065        30.75         10.0   
        A-3.7       N-S   174.083810        20.00         10.0   
        A-4         N-S   118.678605        20.00         10.0   
        A-5         N-S   117.703076        30.75         10.0   
        A-5.1       N-S   211.524369        30.75         10.0   
        A-6         N-S   150.966989        30.75         10.0   
        A-6.1       N-S   126.232285        30.75         10.0   
        A-6.8       N-S   138.386407        30.75         10.0   
        A-7         N-S   154.591903        30.75         10.0   
        A-7.8       N-S   150.479225        20.00         10.0   
        A-8         N-S   121.327990        30.75         10.0   
        A-8.2       N-S   105.237105        20.00         10.0   
        A-8.9       N-S   206.620074        30.75         10.0   
        A-9         N-S   188.069046        30.75         10.0   
Level 4 A-2         N-S   536.694237        30.75         10.5   
        A-2.4       N-S   693.061071        30.75         10.5   
        A-3.7       N-S   367.895788        20.00         10.5   
        A-4         N-S   250.806545        20.00         10.5   
        A-5         N-S   248.744936        30.75         10.5   
        A-5.1       N-S   447.019885        30.75         10.5   
        A-6         N-S   319.042418        30.75         10.5   
        A-6.1       N-S   266.769931        30.75         10.5   
        A-6.8       N-S   292.455550        30.75         10.5   
        A-7         N-S   326.703041        30.75         10.5   
        A-7.8       N-S   318.011613        20.00         10.5   
        A-8         N-S   256.405559        30.75         10.5   
        A-8.2       N-S   222.400279        20.00         10.5   
        A-8.9       N-S   436.655512        30.75         10.5   
        A-9         N-S   397.451147        30.75         10.5   
Level 3 A-2         N-S   728.650626        30.75         10.5   
        A-2.4       N-S   940.944301        30.75         10.5   
        A-3.7       N-S   499.478992        20.00         10.5   
        A-4         N-S   340.511102        20.00         10.5   
        A-5         N-S   337.712129        30.75         10.5   
        A-5.1       N-S   606.902956        30.75         10.5   
        A-6         N-S   433.152513        30.75         10.5   
        A-6.1       N-S   362.184022        30.75         10.5   
        A-6.8       N-S   397.056470        30.75         10.5   
        A-7         N-S   443.553068        30.75         10.5   
        A-7.8       N-S   431.753027        20.00         10.5   
        A-8         N-S   348.112684        30.75         10.5   
        A-8.2       N-S   301.944927        20.00         10.5   
        A-8.9       N-S   592.831618        30.75         10.5   
        A-9         N-S   539.605249        30.75         10.5   
Level 2 A-2         N-S   829.582306        30.75         11.5   
        A-2.4       N-S  1071.282609        30.75         11.5   
        A-3.7       N-S   568.666133        20.00         11.5   
        A-4         N-S   387.678230        20.00         11.5   
        A-5         N-S   384.491548        30.75         11.5   
        A-5.1       N-S   690.970317        30.75         11.5   
        A-6         N-S   493.152202        30.75         11.5   
        A-6.1       N-S   412.353254        30.75         11.5   
        A-6.8       N-S   452.056185        30.75         11.5   
        A-7         N-S   504.993427        30.75         11.5   
        A-7.8       N-S   491.558861        20.00         11.5   
        A-8         N-S   396.332773        30.75         11.5   
        A-8.2       N-S   343.769922        20.00         11.5   
        A-8.9       N-S   674.949836        30.75     

#### East-West Direction

In [63]:
rigid_ew["unit shear"] = flexible_ew["unit shear"]
rigid_ew["wall length"] = flexible_ew["wall length"]
rigid_ew["wall height"] = flexible_ew["wall height"]
rigid_ew["shear stiffness"] = flexible_ew["shear stiffness"]
rigid_ew["nail spacing"] = flexible_ew["nail spacing"]
rigid_ew["end post youngs modulus"] = float(E_endpost)
rigid_ew["end post area"] = float(A_endpost)
rigid_ew["Delta_A"] = float(Delta_A)

rigid_ew["delta_sw"] = (
    8
    * rigid_ew["unit shear"]
    * rigid_ew["wall height"] ** 3
    / (rigid_ew["end post youngs modulus"] * rigid_ew["end post area"] * rigid_ew["wall length"])
    + rigid_ew["unit shear"] * rigid_ew["wall height"] / (1000 * rigid_ew["shear stiffness"])
    + rigid_ew["wall height"] * rigid_ew["Delta_A"] / rigid_ew["wall length"]
)

##### Shear Wall Rigidities

With the deflections of each shear wall, along with the force demand of each shear wall, we can obtain the stiffnesses (and therefore, relative stiffnesses) of all the walls.

$$
k = \frac{F_x}{\delta_{sw}}
$$

In [64]:
rigid_ew["cumulative shear"] = flexible_ew["cumulative shear"]
rigid_ew["wall stiffness"] = rigid_ew["cumulative shear"] / rigid_ew["delta_sw"]
rigid_ew["total level stiffness"] = rigid_ew.groupby("Level")["wall stiffness"].transform("sum")
rigid_ew["total level cumulative shear"] = rigid_ew.groupby("Level")["cumulative shear"].transform("sum")
rigid_ew["relative wall stiffness"] = rigid_ew["wall stiffness"] / rigid_ew["total level stiffness"]
rigid_ew["direct shear force"] = rigid_ew["total level cumulative shear"] * rigid_ew["relative wall stiffness"]
rigid_ew

Direction   unit shear  wall length  wall height  \
Level   Wall                                                       
Roof    A-B.1         E-W   252.906261        21.00         10.0   
        A-B.2.5       E-W   264.252580        13.25         10.0   
        A-B.3.5       E-W   279.758037         9.00         10.0   
        A-B.4         E-W   249.159502        12.00         10.0   
        A-B.4.8       E-W   228.892939        11.00         10.0   
...                   ...          ...          ...          ...   
Level 2 A-C.6.5       E-W   708.244575        11.25         11.5   
        A-C.6.9       E-W   906.194038         7.00         11.5   
        A-C.7.5       E-W  1446.951283         6.75         11.5   
        A-C.8.5       E-W  1069.722353        10.25         11.5   
        A-C.9.2       E-W   576.752218        25.25         11.5   

                 shear stiffness  nail spacing  end post youngs modulus  \
Level   Wall                                                              
Roof    A-B.1               14.0           6.0                1600000.0   
        A-B.2.5             14.0           6.0                1600000.0   
        A-B.3.5             14.0           6.0                1600000.0   
        A-B.4               14.0           6.0                1600000.0   
        A-B.4.8             14.0           6.0                1600000.0   
...                          ...           ...                      ...   
Level 2 A-C.6.5             19.0           3.0                1600000.0   
        A-C.6.9             23.0           2.0                1600000.0   
        A-C.7.5             19.0           3.0                1600000.0   
        A-C.8.5             23.0           2.0                1600000.0   
        A-C.9.2             17.0           4.0                1600000.0   

                 end post area  Delta_A  delta_sw  cumulative shear  \
Level   Wall                                                          
Roof    A-B.1             10.5     0.25  0.305430       5311.031480   
        A-B.2.5           10.5     0.25  0.386928       3501.346680   
        A-B.3.5           10.5     0.25  0.492407       2517.822331   
        A-B.4             10.5     0.25  0.396192       2989.914019   
        A-B.4.8           10.5     0.25  0.400676       2517.822331   
...                        ...      ...       ...               ...   
Level 2 A-C.6.5           10.5     0.25  0.729824       7967.751471   
        A-C.6.9           10.5     0.25  0.957567       6343.358268   
        A-C.7.5           10.5     0.25  1.456960       9766.921158   
        A-C.8.5           10.5     0.25  0.890932      10964.654121   
        A-C.9.2           10.5     0.25  0.520560      14562.993495   

                 wall stiffness  total level stiffness  \
Level   Wall                                             
Roof    A-B.1      17388.715036          215842.289617   
        A-B.2.5     9049.090153          215842.289617   
        A-B.3.5     5113.295584          215842.289617   
        A-B.4       7546.634903          215842.289617   
        A-B.4.8     6283.928620          215842.289617   
...                         ...                    ...   
Level 2 A-C.6.5    10917.365908          327888.366997   
        A-C.6.9     6624.454117          327888.366997   
        A-C.7.5     6703.631934          327888.366997   
        A-C.8.5    12306.955097          327888.366997   
        A-C.9.2    27975.637980          327888.366997   

                 total level cumulative shear  relative wall stiffness  \
Level   Wall                                                             
Roof    A-B.1                    72387.392027                 0.080562   
        A-B.2.5                  72387.392027                 0.041925   
        A-B.3.5                  72387.392027                 0.023690   
        A-B.4                    72387.392027                 0.034964   
        A-B.4.8                 

##### Story Drift

We can check the story drift at each shear wall line. This is conservative but all-encompassing.

The story drift is defined as:

$$
\delta_x = \frac{C_d \delta_{xe}}{I_e}\quad \text{(ASCE 7-16 Eq. 12.8-15)}
$$

We can get the elastic story displacement, $\delta_{xe}$, by getting a displacement for each shear line (which will be conservative). Since we got a stiffness using $\delta_{sw}$, we can add those stiffnesses up for each line then get a displacement with $u = F/k$.

The story drift can't exceed the allowable story drift, $\Delta_a$, which is defined in ASCE 7-16 Table 12.12-1. In our case:

- Structure is not masonry shear wall
- Four stories or less above the base
- Interior walls, partitions, ceilings, and exterior wall systems are designed to accommodate story drifts.

Therefore:

$$
\Delta_a = 0.025 h_{sx}
$$

where $h_{sx}$ is the story height below level $x$.

In [65]:
rigid_ew["story drift"] = rigid_ew["delta_sw"] * C_d / I_e
rigid_ew["Delta_a"] = 0.025 * rigid_ew["wall height"] * 12
rigid_ew["story drift check"] = rigid_ew["story drift"] <= rigid_ew["Delta_a"]
rigid_ew["story drift dcr"] = rigid_ew["story drift"] / rigid_ew["Delta_a"]

The ratio of the story drift, $\Delta_x$, and the the maximum story drift, $\Delta_a$, must not exceed 1

$$
\text{ratio}_{drift} = \frac{\Delta_x}{\Delta_a} \leq 1
$$

In [66]:
%%render
ratio_drift__max = rigid_ew["story drift dcr"].max()
check_Delta = check_value(ratio_drift__max, 1, inequality="<=")

<IPython.core.display.Latex object>

In [67]:
if rigid_ew["story drift check"].all():
    display_text("All story drifts are OK")

In [68]:
display_table(
    rigid_ew.loc[
        rigid_ew["story drift dcr"] > 1,
        [
            "shear stiffness",
            "story drift dcr",
        ],
    ],
    caption="\\textbf{\\color{red}SHEAR WALLS WITH INSUFFICIENT STIFFNESS}",
)

shear stiffness  story drift dcr
Level   Wall                                     
Level 4 A-C.7.5             23.0         1.133570
Level 3 A-C.1.8             17.0         1.143241
        A-C.6.9             19.0         1.114340
        A-C.7.5             17.0         1.622415
Level 2 A-B.6.5             17.0         1.229758
        A-B.8.8             17.0         1.245879
        A-C.1.8             17.0         1.285451
        A-C.6.9             23.0         1.110223
        A-C.7.5             19.0         1.689229
        A-C.8.5             23.0         1.032964